# **Pandas en Python: Trabajando Agrupaciones Multiples, Pivotar y Transponer**

### **Importar librería**

In [2]:
import pandas as pd

### **Carga de Datos**

In [3]:
# CARGA DE DATOS
df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/Sistemas Big Data/df_final.csv") # Por defecto detecta las columnas por el encabezado del archivo
df.head()

,Unnamed: 0,id,full_text,favorites,retweets,mentions,user,fecha_publicado,ciudad,followers,followees,pais,fecha_pulblicacion_norm,dia_semana,dia_semana_fun,mes
0,0,183721,Flying home to run down from the power to comi...,23.0,NaN,10.0,leonardokuffo,27/01/2023,GUAYAQUIL,389.0,258,ECUADOR,2023-01-27,Friday,Friday,01/2023
1,1,183722,Today we commemorate and MNML Case.,500.0,21.0,NaN,mateusmartins,27/01/2023,SAO PAULO,982.0,1822,BRASIL,2023-01-27,Friday,Friday,01/2023
2,2,183723,Today we have reached US$6.55 Billion TT$44…,190.0,123.0,6.0,pedrojuarez,28/01/2023,OAXACA,12.0,129,MEXICO,2023-01-28,Saturday,Saturday,01/2023
3,3,183724,Faking It by Joel Atwell. Written by Other cou...,131.0,76.0,3.0,galocastillo,28/01/2023,QUITO,332.0,378,ECUADOR,2023-01-28,Saturday,Saturday,01/2023
4,4,183725,Welcome back! 🙌,113.0,130.0,9.0,pedrojuarez,28/01/2023,OAXACA,12.0,129,MEXICO,2023-01-28,Saturday,Saturday,01/2023


# **AGRUPACIONES MULTIPLES**

### Agrupación simple
Ejemplo por mes

In [7]:
agrupado = df.groupby(['mes']).agg({
    'id' : 'count'
}).rename(columns={'id': 'nº tweets'})
agrupado


,nº tweets
mes,
01/2023,12
02/2023,12
03/2023,3


### Agrupar por multiples niveles en mis dataframes
Ejemplo por mes y ciudad

In [16]:
agrupado = df.groupby(['mes', 'ciudad']).agg({
    'id' : 'count'
}).rename(columns={'id': 'nº tweets'})
agrupado
# El nivel 0 sera mes, y el nivel 1 sera ciudad

nº tweets
mes     ciudad                   
01/2023 GUAYAQUIL               2
        OAXACA                  4
        QUITO                   1
        SAO PAULO               5
02/2023 GUADALAJARA             2
        GUAYAQUIL               2
        OAXACA                  1
        QUITO                   3
        RIO DE JANEIRO          4
03/2023 GUADALAJARA             1
        PORTO ALEGRE            1
        RIO DE JANEIRO          1

#### Reseteo de un Indice (nivel) para que se comporte como una columna (desagrupar)



In [17]:
agrupado["cuidad"]

KeyError: 'cuidad'

In [18]:
desagrupado = agrupado.reset_index(level=1)
# desagrupado = agrupado.reset_index(1)

In [19]:
desagrupado["ciudad"]

mes
01/2023         GUAYAQUIL
01/2023            OAXACA
01/2023             QUITO
01/2023         SAO PAULO
02/2023       GUADALAJARA
02/2023         GUAYAQUIL
02/2023            OAXACA
02/2023             QUITO
02/2023    RIO DE JANEIRO
03/2023       GUADALAJARA
03/2023      PORTO ALEGRE
03/2023    RIO DE JANEIRO
Name: ciudad, dtype: object

# **PIVOTAR**

[Doc Pivotar](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.pivot.html)

In [20]:
pivote = desagrupado.pivot(columns="ciudad")
pivote

nº tweets                                                     \
ciudad  GUADALAJARA GUAYAQUIL OAXACA PORTO ALEGRE QUITO RIO DE JANEIRO   
mes                                                                      
01/2023         NaN       2.0    4.0          NaN   1.0            NaN   
02/2023         2.0       2.0    1.0          NaN   3.0            4.0   
03/2023         1.0       NaN    NaN          1.0   NaN            1.0   

                   
ciudad  SAO PAULO  
mes                
01/2023       5.0  
02/2023       NaN  
03/2023       NaN

#### Extraer los datos de una columna

In [11]:
pivote[["GUAYAQUIL"]]
# Da error, ya que al igual que en agrupado, aqui tenemos niveles, el 0 que seria id(nº de tweets) y el 1 que es ciudad

NameError: name 'pivote' is not defined

In [21]:
pivote = pivote.droplevel(level =0, axis = "columns")
pivote

ciudad,GUADALAJARA,GUAYAQUIL,OAXACA,PORTO ALEGRE,QUITO,RIO DE JANEIRO,SAO PAULO
mes,,,,,,,
01/2023,NaN,2.0,4.0,NaN,1.0,NaN,5.0
02/2023,2.0,2.0,1.0,NaN,3.0,4.0,NaN
03/2023,1.0,NaN,NaN,1.0,NaN,1.0,NaN


In [22]:
pivote[["GUAYAQUIL"]]

ciudad,GUAYAQUIL
mes,
01/2023,2.0
02/2023,2.0
03/2023,NaN


In [23]:
# Rellenar a 0
pivote = pivote.fillna(0)
pivote

ciudad,GUADALAJARA,GUAYAQUIL,OAXACA,PORTO ALEGRE,QUITO,RIO DE JANEIRO,SAO PAULO
mes,,,,,,,
01/2023,0.0,2.0,4.0,0.0,1.0,0.0,5.0
02/2023,2.0,2.0,1.0,0.0,3.0,4.0,0.0
03/2023,1.0,0.0,0.0,1.0,0.0,1.0,0.0


# **TRANSPONER**
[Doc Transponer](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.transpose.html)

In [ ]:
pivote.transpose()

mes,01/2023,02/2023,03/2023
ciudad,,,
GUADALAJARA,0.0,2.0,1.0
GUAYAQUIL,2.0,2.0,0.0
OAXACA,4.0,1.0,0.0
PORTO ALEGRE,0.0,0.0,1.0
QUITO,1.0,3.0,0.0
RIO DE JANEIRO,0.0,4.0,1.0
SAO PAULO,5.0,0.0,0.0
